# 04 · Fire-activity forecast: when will the 2026 season peak, and how high?

**Project:** Paddy Burnt-Area Intelligence (Satellite Mission Intelligence for India)
**Question for this notebook:** from the 2026 season so far, how many more VIIRS fire detections will Punjab and Sangrur district have before 30 November, and in which week will they peak?

**Why:** a straw buyer has to get balers into fields before the straw is burnt, and an agriculture department has to place machines and enforcement teams before the busiest week. Both need to know, early, when the season will peak and roughly how big it will be. Detections have also fallen about tenfold since 2021, so last year's numbers alone are a poor guide.

**How it works**
1. **Weekly counts.** Suomi NPP detections are counted by week: 11 whole weeks from 15 Sep to 30 Nov, as in Notebook 01, for Punjab and for Sangrur (pre-2021 boundary). Seasons 2012–2025 come from Notebook 01's standard-processing archive; 2026 is downloaded fresh, in near real time.
2. **Three methods** forecast the rest of the season from the weeks seen so far:
   - *last season*: each remaining week as in the same week of 2025 (the seasonal-naive baseline);
   - *SARIMA*: a seasonal ARIMA model of the log counts;
   - *past seasons rescaled*: each season since 2012 keeps its own week-by-week shape but is scaled to this season's level so far. The middle scenario is the forecast.
3. **Backtests.** Each method forecasts every season from 2016 to 2025 from the same point in the season as the 2026 forecast, using only earlier data. The method with the smallest typical error over the last five seasons becomes the headline, and the spread of its errors over all ten seasons gives an 80% range.
4. **The 2026 forecast is made once.** It is saved with the date it was made and is not overwritten by later runs, so it can be scored honestly.
5. **The check.** As the weeks pass, later runs compare the forecast with what happened. Suomi NPP data stop at 13:00 UTC on 2 Nov 2026, so from 3 Nov the actual counts come from NOAA-20, converted with the ratio of the two sensors earlier in the 2026 season.

**Before you run it:** run Notebook 01 (its detections file and district boundaries are reused), and keep `FIRMS_MAP_KEY` in `.env` as for Notebook 01. Nothing is printed from `.env`.

**Run time:** about a minute, plus the first download of the 2026 detections (some 20 small requests; finished periods are cached in `data/raw/firms/`).

**What it produces**

| Output | Where |
|---|---|
| Weekly detections by area, season and sensor, 2012–2026 | `data/processed/fire_weekly_counts.csv` |
| Backtests of the three methods, 2016–2025 | `data/processed/fire_forecast_backtests.csv` |
| The 2026 forecast, week by week, as issued | `data/processed/fire_forecast_2026.csv` |
| The 2026 forecast in brief: rest of season, range, peak week, as issued | `data/processed/fire_forecast_2026_summary.csv` |
| The forecast against what happened, refreshed at each run | `data/processed/fire_forecast_check_2026.csv` |
| Figures | `maps/notebook04/` |
| Source register rows for the 2026 downloads | `sources/source_register.csv` |

**Read the numbers carefully.** This forecasts fire *detections*: 375 m pixels with fire at the moment a satellite passed, mostly in the early afternoon. It does not forecast burnt area, which Notebook 02 maps.

In [ ]:
# Colab: add libraries its default image may lack. On your own computer, install requirements.txt into the
# project's environment first; there this cell only checks that the install worked.
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401  (this module exists only on Colab)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

NEEDED = ('geopandas', 'pyarrow', 'statsmodels', 'matplotlib', 'requests', 'pandas')
if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'geopandas', 'pyarrow', 'statsmodels'], check=True)
else:
    missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError('Missing: ' + ', '.join(missing) + ". Install requirements.txt into the "
                                  "project's environment and re-run.")
    print('Libraries found.')

## 1. Settings

In [ ]:
import os, io, re, time, inspect, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import geopandas as gpd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from IPython.display import display

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/India-Space-Mission-Intelligence')
elif os.environ.get('PROJECT_DIR'):
    PROJECT = Path(os.environ['PROJECT_DIR']).resolve()
else:
    # On your computer: walk up from the working folder to the project root (the folder with requirements.txt).
    here = Path.cwd().resolve()
    PROJECT = next((p for p in (here, *here.parents)
                    if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), here.parent)

RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
FIGS = PROJECT / 'maps' / 'notebook04'
SOURCES = PROJECT / 'sources'
for folder in (RAW / 'firms', PROCESSED, FIGS, SOURCES):
    folder.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
print('Project folder:', PROJECT)

# --- When -------------------------------------------------------------------------------------------------
TODAY = pd.Timestamp.now(tz='Asia/Kolkata').tz_localize(None).normalize()   # today's date in India
SEASON = 2026
SEASON_START = '09-15'          # as in Notebook 01: the season runs from 15 Sep to 30 Nov, 11 whole weeks
WEEKS = 11
SNPP_LAST_DAY = '2026-11-02'    # NOAA stops Suomi NPP data at 13:00 UTC (18:30 IST) on 2 Nov 2026

# --- Where ------------------------------------------------------------------------------------------------
AREAS = {'Punjab': ('state', 'Punjab'),
         'Sangrur': ('district', 'Sangrur')}    # Sangrur: the pre-2021 district, Malerkotla included
STATES = ['Punjab', 'Haryana']                  # the states Notebook 01 covers
BBOX = (73.8, 27.6, 77.6, 32.6)                 # west, south, east, north: Notebook 01's download box

# --- The methods and their backtests ------------------------------------------------------------------------
FIRST_SEASON = 2012                             # first season of the Suomi NPP archive
BACKTEST_SEASONS = list(range(2016, SEASON))    # each forecast from at least four earlier seasons
RECENT_SEASONS = 5                              # the headline method does best over the last 5 backtests
SARIMA_ORDER = (1, 0, 0)                        # this week's departure from last season carries on ...
SARIMA_SEASONAL_ORDER = (0, 1, 1)               # ... while a seasonal moving average smooths year-to-year noise
SIMULATIONS = 1000                              # SARIMA paths drawn for the peak week and the totals
SEED = 0
INTERVAL = 0.8                                  # the range given for the rest of the season, from past errors

# --- Downloads (as in Notebook 01) ----------------------------------------------------------------------
DAY_RANGE = 5                                   # days per FIRMS request (the API accepts 1-5)
PAUSE_S = 1.0                                   # pause after each live request

REISSUE = False     # True replaces a forecast already saved. Leave it False: the forecast is made once.

## 2. Helper functions

**Downloads and cleaning** repeat Notebook 01's rules, so 2026 counts compare with the archive: duplicates dropped, the "presumed vegetation fire" type kept where the archive gives it, time converted to IST, and points joined to the same geoBoundaries districts.

**The three methods.** Each takes the earlier seasons (one row of 11 weekly counts per season) and the weeks seen so far this season, and returns *scenarios* for the remaining weeks:
- **last season** returns one scenario: the remaining weeks of the previous season.
- **SARIMA** fits a seasonal ARIMA model to the log of (1 + weekly count), with the seasons laid end to end and a season of 11 weeks as the period, and draws 1,000 possible paths.
- **past seasons rescaled** returns one scenario per earlier season: that season's remaining weeks, multiplied by this season's detections so far divided by that season's detections over the same weeks.

From the scenarios come the forecast for each week (the median), the rest-of-season total (the median of the scenario totals), the most likely peak week (the week most scenarios peak in, counting the weeks already seen) and how often scenarios peak there.

**Typical error factor.** In the backtests, each forecast of the rest-of-season total is compared with what happened as a ratio of at least 1: forecast over actual, or actual over forecast, whichever is larger. The typical error factor is the geometric mean of those ratios, so 1.5 means forecasts were typically 1.5 times too high or too low. One is added to both totals, so a season with no detections left can't divide by zero.

In [ ]:
FIRMS_API = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{source}/{bbox}/{days}/{date}'
REGISTER = SOURCES / 'source_register.csv'
REGISTER_COLUMNS = ['source_id', 'dataset_or_product', 'provider', 'url', 'date_accessed', 'geography',
                    'time_range', 'resolution', 'licence_or_terms', 'preprocessing', 'known_limitations', 'used_in']
SOURCE_SENSOR = {'VIIRS_SNPP_NRT': 'SNPP', 'VIIRS_NOAA20_NRT': 'NOAA20'}
SENSOR_NAME = {'SNPP': 'Suomi NPP', 'NOAA20': 'NOAA-20'}


def season_start(season):
    return pd.Timestamp(f'{season}-{SEASON_START}')


def week_dates(season, week):
    start = season_start(season) + pd.Timedelta(7 * (int(week) - 1), unit='D')
    return start, start + pd.Timedelta(6, unit='D')


def day_month(day):
    return f'{day.day} {day:%b}'


def week_label(season, week):
    start, end = week_dates(season, week)
    return f'week {week} ({day_month(start)}–{day_month(end)})'


def read_env_file(path):
    # KEY=value lines from a .env file; blank lines and lines starting with # are skipped.
    values = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8-sig').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                name, value = line.split('=', 1)
                values[name.strip()] = value.strip().strip('"\'')
    return values


def get_map_key():
    key = None
    if IN_COLAB:
        try:
            key = userdata.get('FIRMS_MAP_KEY')
        except Exception:
            key = None
    key = key or os.environ.get('FIRMS_MAP_KEY') or read_env_file(PROJECT / '.env').get('FIRMS_MAP_KEY')
    if not key:
        raise RuntimeError('No FIRMS map key found. As for Notebook 01: on your computer, put the line '
                           f'FIRMS_MAP_KEY=your_key in a file named .env in {PROJECT}; in Colab, add a secret '
                           'named FIRMS_MAP_KEY. Then re-run this cell.')
    return key.strip()


def http_get(url, timeout=120):
    return requests.get(url, timeout=timeout)


def register_source(**row):
    # Add or replace one row of the source register, keyed by source_id.
    row = {c: str(row.get(c, '')) for c in REGISTER_COLUMNS}
    reg = pd.read_csv(REGISTER, dtype=str).fillna('') if REGISTER.exists() else pd.DataFrame(columns=REGISTER_COLUMNS)
    reg = reg[reg['source_id'] != row['source_id']]
    reg = pd.concat([reg, pd.DataFrame([row])], ignore_index=True)
    reg[REGISTER_COLUMNS].to_csv(REGISTER, index=False)


class DayRangeTooLong(Exception):
    # FIRMS refused the number of days in one request; args[0] is the most it accepts.
    pass


def fetch_chunk(source, start, days):
    # One block of `days` days of detections in the box; complete periods are cached, as in Notebook 01.
    start_ts = pd.Timestamp(start)
    end_ts = start_ts + pd.Timedelta(days - 1, unit='D')
    cache = RAW / 'firms' / source / f'{start_ts:%Y-%m-%d}_{days}d.csv'
    if cache.exists():
        return pd.read_csv(cache)
    url = FIRMS_API.format(key=MAP_KEY, source=source, bbox=','.join(map(str, BBOX)), days=days,
                           date=f'{start_ts:%Y-%m-%d}')
    waits = [60, 120, 300, 600]
    message = ''
    for attempt in range(len(waits) + 1):
        try:
            r = http_get(url)
            text = r.text.strip()
            if r.status_code == 200 and (text == '' or text.startswith('latitude')):
                df = pd.read_csv(io.StringIO(text)) if text else pd.DataFrame(columns=['latitude', 'longitude'])
                if end_ts < TODAY - pd.Timedelta(2, unit='D'):     # cache only periods that are complete
                    cache.parent.mkdir(parents=True, exist_ok=True)
                    df.to_csv(cache, index=False)
                time.sleep(PAUSE_S)
                return df
            message = text[:200]
        except requests.RequestException as err:
            message = str(err).replace(MAP_KEY, '***')[:200]    # requests puts the URL, key included, in errors
        limit = re.search(r'day range.*?\[1\s*\.\.\s*(\d+)\]', message, flags=re.I)
        if limit:
            raise DayRangeTooLong(int(limit.group(1)))
        if 'invalid' in message.lower():
            hint = ' Check that FIRMS_MAP_KEY holds the key from your FIRMS email.' if 'key' in message.lower() else ''
            raise RuntimeError(f'FIRMS rejected the request: {message}.{hint}')
        if attempt < len(waits):
            print(f'  {source} {start_ts:%Y-%m-%d}: {message!r}; retrying in {waits[attempt]} s')
            time.sleep(waits[attempt])
    raise RuntimeError(f'Could not download {source} from {start_ts:%Y-%m-%d}: {message}')


def download(source, last_day):
    # Detections from the start of the season to last_day, in Notebook 01's blocks (so its cache is reused).
    global DAY_RANGE
    frames, day = [], season_start(SEASON)
    while day <= last_day:
        days = int(min(DAY_RANGE, (last_day - day).days + 1))
        try:
            chunk = fetch_chunk(source, day, days)
        except DayRangeTooLong as err:
            if not 1 <= err.args[0] < days:
                raise RuntimeError(f'FIRMS refused a {days}-day request but says it accepts up to {err.args[0]}.')
            DAY_RANGE = err.args[0]
            continue
        if len(chunk):
            frames.append(chunk.assign(source=source))
        day += pd.Timedelta(days, unit='D')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def plain_name(name):
    # Lower-case ASCII form of a place name, so geoBoundaries' 'Haryāna' matches 'Haryana'.
    return unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode('ascii').strip().lower()


def load_districts():
    # The district polygons Notebook 01 downloaded, assigned to a state by their representative point as there.
    paths = [RAW / 'boundaries' / f'geoBoundaries-IND-{adm}.geojson' for adm in ('ADM1', 'ADM2')]
    if not all(p.exists() for p in paths):
        raise RuntimeError(f'District boundaries not found in {RAW / "boundaries"}: run Notebook 01 first.')
    adm1, adm2 = (gpd.read_file(p).to_crs(4326) for p in paths)
    spelling = {plain_name(s): s for s in STATES}
    states = adm1.assign(state=adm1['shapeName'].map(plain_name).map(spelling))
    states = states.loc[states['state'].notna(), ['state', 'geometry']]
    if 'shapeID' not in adm2.columns:
        adm2['shapeID'] = adm2.index.astype(str)
    centres = gpd.GeoDataFrame(adm2[['shapeID']].copy(), geometry=adm2.representative_point(), crs=4326)
    in_states = gpd.sjoin(centres, states, predicate='within', how='inner')[['shapeID', 'state']]
    return (adm2.merge(in_states, on='shapeID')
                .rename(columns={'shapeName': 'district'})[['district', 'state', 'geometry']])


def clean(raw, districts):
    # Notebook 01's cleaning rules; returns one row per detection with its IST date, sensor and district.
    df = raw.copy()
    df['acq_time'] = pd.to_numeric(df['acq_time'], errors='coerce')
    df = df.dropna(subset=['latitude', 'longitude', 'acq_date', 'acq_time'])
    clock = df['acq_time'].astype(int).astype(str).str.zfill(4)
    df['time_utc'] = pd.to_datetime(df['acq_date'].astype(str) + ' ' + clock, format='%Y-%m-%d %H%M',
                                    utc=True, errors='coerce')
    df = df.dropna(subset=['time_utc'])
    df['date_ist'] = df['time_utc'].dt.tz_convert('Asia/Kolkata').dt.tz_localize(None).dt.normalize()
    df = df.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time', 'source'])
    if 'type' in df.columns:
        df = df[df['type'].isna() | (df['type'] == 0)]
    df['sensor'] = df['source'].map(SOURCE_SENSOR)
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['longitude'], df['latitude']), crs=4326)
    pts = gpd.sjoin(pts, districts, predicate='within', how='inner')
    return pd.DataFrame(pts[['date_ist', 'sensor', 'district', 'state']]).assign(season=SEASON)


def weekly_table(points, sensor, area=None):
    # Detections per season (rows) and week 1-11 (columns), for one sensor, in one area (None: both states).
    p = points[points['sensor'] == sensor]
    if area is not None:
        column, name = AREAS[area]
        p = p[(p[column] == name) & (p['state'] == 'Punjab')]
    starts = pd.to_datetime(p['season'].astype(str) + '-' + SEASON_START)
    week = (pd.to_datetime(p['date_ist']) - starts).dt.days // 7 + 1
    inside = week.between(1, WEEKS)
    table = p[inside].groupby([p.loc[inside, 'season'], week[inside]]).size().unstack(fill_value=0)
    first = FIRST_SEASON if sensor == 'SNPP' else 2020           # NOAA-20's archive starts in 2020
    return table.reindex(index=range(first, SEASON + 1), columns=range(1, WEEKS + 1), fill_value=0).astype(float)


# --- The three methods: each returns scenarios for the remaining weeks (one row per scenario) ---------------
def last_season(past, seen, issue):
    return past[-1:, issue:].astype(float)


def sarima(past, seen, issue):
    y = np.log1p(np.concatenate([past.ravel(), seen[:issue]]).astype(float))
    steps = WEEKS - issue
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fit = SARIMAX(y, order=SARIMA_ORDER, seasonal_order=(*SARIMA_SEASONAL_ORDER, WEEKS), trend='n').fit(disp=False)
        # statsmodels 0.15 seeds simulations with rng=, earlier versions with random_state=.
        seed = ({'rng': np.random.default_rng(SEED)} if 'rng' in inspect.signature(fit.simulate).parameters
                else {'random_state': SEED})
        paths = fit.simulate(nsimulations=steps, repetitions=SIMULATIONS, anchor='end', **seed)
    return np.clip(np.expm1(np.asarray(paths).reshape(steps, SIMULATIONS).T), 0, None)


def past_seasons_rescaled(past, seen, issue):
    level = seen[:issue].sum() / np.maximum(past[:, :issue].sum(axis=1), 1)
    return past[:, issue:] * level[:, None]


METHODS = {'last season': last_season, 'SARIMA': sarima, 'past seasons rescaled': past_seasons_rescaled}
COLOURS = {'last season': 'tab:gray', 'SARIMA': 'tab:blue', 'past seasons rescaled': 'tab:orange'}


def summarise(scenarios, seen):
    # Weekly forecast, rest-of-season total and the peak, from a set of scenarios.
    full = np.hstack([np.repeat(seen[None, :], len(scenarios), axis=0), scenarios])
    peaks = np.bincount(full.argmax(axis=1) + 1, minlength=WEEKS + 1)
    peak_week = int(peaks.argmax())
    return {'weekly': np.median(scenarios, axis=0), 'rest': float(np.median(scenarios.sum(axis=1))),
            'peak_week': peak_week, 'peak_share': peaks[peak_week] / len(scenarios),
            'peak_height': float(np.median(full.max(axis=1)))}


def log_ratio(forecast, actual):
    return np.log((np.asarray(forecast, float) + 1) / (np.asarray(actual, float) + 1))

## 3. Weekly detections, 2012 to 2026

Seasons up to 2025 come from Notebook 01's detections file (standard processing). The 2026 season so far is downloaded now, in near real time, for Suomi NPP (until it stops on 2 Nov) and NOAA-20. Suomi NPP is the forecast's sensor, because its record goes back to 2012.

The forecast starts after the last whole week of Suomi NPP data: on 1 or 2 November that is week 6 (20–26 Oct); from 3 November, week 7 (27 Oct–2 Nov), the last week Suomi NPP covers. A forecast already saved keeps the week it was made from.

In [ ]:
archive_file = PROCESSED / 'firms_viirs_punjab_haryana_points.parquet'
if not archive_file.exists():
    raise RuntimeError(f'{archive_file.name} not found: run Notebook 01 first.')
archive = pd.read_parquet(archive_file, columns=['date_ist', 'season', 'sensor', 'product', 'district', 'state'])
archive = archive[(archive['product'] == 'SP') & (archive['season'] < SEASON)]
missing = sorted(set(range(FIRST_SEASON, SEASON)) - set(archive.loc[archive['sensor'] == 'SNPP', 'season']))
if missing:
    raise RuntimeError(f"{archive_file.name} has no Suomi NPP seasons {missing}: re-run Notebook 01.")

MAP_KEY = get_map_key()
print('Map key loaded (not printed).')
districts = load_districts()
season_end = week_dates(SEASON, WEEKS)[1]                # 30 Nov
last_full_day = min(TODAY - pd.Timedelta(1, unit='D'), season_end)     # today's passes may not all be in yet
snpp_end = min(last_full_day, pd.Timestamp(SNPP_LAST_DAY))
raw_2026 = pd.concat([download('VIIRS_SNPP_NRT', snpp_end), download('VIIRS_NOAA20_NRT', last_full_day)],
                     ignore_index=True)
if raw_2026.empty:
    raise RuntimeError('No 2026 detections downloaded. Check the map key, and the FIRMS status page if it persists.')
live = clean(raw_2026, districts)
live = live[live['date_ist'] <= last_full_day]
points = pd.concat([archive, live], ignore_index=True)
print(f'{SEASON} so far (to {last_full_day:%d %b}): ' + ', '.join(
    f'{SENSOR_NAME[s]} {n:,}' for s, n in live.groupby('sensor').size().items()) + ' detections in Punjab and Haryana.')

snpp = {area: weekly_table(points, 'SNPP', area) for area in AREAS}
noaa20 = {area: weekly_table(points, 'NOAA20', area) for area in AREAS}

# Which weeks of 2026 are complete, for each sensor.
week_ends = pd.Series({w: week_dates(SEASON, w)[1] for w in range(1, WEEKS + 1)})
complete = {'SNPP': week_ends <= snpp_end, 'NOAA20': week_ends <= last_full_day}

FORECAST_FILE = PROCESSED / 'fire_forecast_2026.csv'
SUMMARY_FILE = PROCESSED / 'fire_forecast_2026_summary.csv'
frozen = FORECAST_FILE.exists() and SUMMARY_FILE.exists() and not REISSUE
if frozen:
    saved_summary = pd.read_csv(SUMMARY_FILE)
    ISSUE_WEEK = int(saved_summary['issue_week'].iloc[0])
    print(f"A forecast was saved on {saved_summary['issued_on'].iloc[0]}, from weeks 1-{ISSUE_WEEK}: it is kept.")
else:
    ISSUE_WEEK = int(complete['SNPP'].sum())
    print(f'Suomi NPP data are complete to the end of {week_label(SEASON, ISSUE_WEEK)}; '
          f'the forecast starts from there.' if ISSUE_WEEK else 'No complete week of Suomi NPP data yet.')

# The ratio of the two sensors, to express NOAA-20 counts in Suomi NPP terms once Suomi NPP stops (both states).
both = {s: weekly_table(points, s) for s in ('SNPP', 'NOAA20')}
overlap = complete['SNPP'] & complete['NOAA20']
ratio_2026 = (both['SNPP'].loc[SEASON, overlap.to_numpy()].sum()
              / max(both['NOAA20'].loc[SEASON, overlap.to_numpy()].sum(), 1))
ratio_file = PROCESSED / 'firms_snpp_noaa20_ratio.csv'
past_ratio = pd.read_csv(ratio_file)['snpp_per_noaa20'] if ratio_file.exists() else pd.Series(dtype=float)
print(f'Suomi NPP per NOAA-20 detection, {SEASON} weeks 1-{int(overlap.sum())}: {ratio_2026:.2f}'
      + (f' (2020-2025 seasons: {past_ratio.min():.2f} to {past_ratio.max():.2f})' if len(past_ratio) else ''))

# Week by week, the two sensors should see similar numbers; a sudden gap would mean missing Suomi NPP data.
by_week = pd.DataFrame({'Suomi NPP': both['SNPP'].loc[SEASON], 'NOAA-20': both['NOAA20'].loc[SEASON]})
by_week = by_week[overlap.to_numpy()]
by_week['ratio'] = (by_week['Suomi NPP'] / by_week['NOAA-20'].replace(0, np.nan)).round(2)
display(by_week.T)
for week, r in by_week[(by_week['NOAA-20'] >= 50) & ~by_week['ratio'].between(0.6, 1.5)].iterrows():
    print(f"WARNING: in {week_label(SEASON, week)} Suomi NPP saw {r['ratio']:.0%} as many detections as NOAA-20. "
          'Check for missing Suomi NPP data before relying on the forecast.')

fig, axes = plt.subplots(1, len(AREAS), figsize=(12, 4.2))
for ax, area in zip(np.atleast_1d(axes), AREAS):
    table = snpp[area]
    for season in table.index[table.index < SEASON]:
        recent = season >= SEASON - 5
        ax.plot(range(1, WEEKS + 1), table.loc[season] + 1, color=plt.cm.viridis((season - SEASON + 6) / 6)
                if recent else '0.8', linewidth=1.6 if recent else 0.8, label=str(season) if recent else None)
    seen = complete['SNPP'].to_numpy()
    ax.plot(np.arange(1, WEEKS + 1)[seen], table.loc[SEASON].to_numpy()[seen] + 1, color='k', linewidth=2.4,
            marker='o', label=f'{SEASON} so far')
    ax.set_yscale('log')
    ax.set_xticks(range(1, WEEKS + 1))
    ax.set_xticklabels([day_month(week_dates(SEASON, w)[0]) for w in range(1, WEEKS + 1)], rotation=45, fontsize=8)
    ax.set_title(f'{area}: Suomi NPP detections per week (grey: 2012 onward)', fontsize=10)
    ax.set_ylabel('detections + 1 (log scale)')
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / '01_weekly_detections.png', dpi=150)
plt.show()

## 4. Backtests: how each method would have done

Each season from 2016 to 2025 is forecast from the same week as the 2026 forecast, with only the seasons before it and its own weeks up to then. The table gives each method's typical error factor for the rest-of-season total over all ten seasons and over the last five, how often it found the peak week, and how often it was within a week.

The headline method for each area is the one with the smallest typical error over the last five seasons, when detections were falling fastest. Its errors in all ten seasons set the 80% range given with the forecast.

In [ ]:
if not 1 <= ISSUE_WEEK < WEEKS:
    raise RuntimeError(f'A forecast needs between 1 and {WEEKS - 1} whole weeks of the season; there are {ISSUE_WEEK}.')
rows, backtest_weekly = [], {}
for area in AREAS:
    table = snpp[area]
    for season in BACKTEST_SEASONS:
        past = table.loc[FIRST_SEASON:season - 1].to_numpy(float)
        actual = table.loc[season].to_numpy(float)
        for name, method in METHODS.items():
            s = summarise(method(past, actual[:ISSUE_WEEK], ISSUE_WEEK), actual[:ISSUE_WEEK])
            backtest_weekly[area, season, name] = s['weekly']
            rows.append({'area': area, 'issue_week': ISSUE_WEEK, 'season': season, 'method': name,
                         'actual_rest': actual[ISSUE_WEEK:].sum(), 'forecast_rest': round(s['rest'], 1),
                         'actual_peak_week': int(actual.argmax()) + 1, 'forecast_peak_week': s['peak_week']})
backtests = pd.DataFrame(rows)
backtests['error_factor'] = np.exp(np.abs(log_ratio(backtests['forecast_rest'], backtests['actual_rest'])))
backtests['peak_week_right'] = backtests['actual_peak_week'] == backtests['forecast_peak_week']
backtests['peak_week_within_1'] = (backtests['actual_peak_week'] - backtests['forecast_peak_week']).abs() <= 1

score_rows = []
for (area, name), g in backtests.groupby(['area', 'method']):
    errors = np.abs(log_ratio(g['forecast_rest'], g['actual_rest']))
    last = (g['season'] >= BACKTEST_SEASONS[-RECENT_SEASONS]).to_numpy()
    score_rows.append({'area': area, 'method': name, 'typical_error_all': np.exp(errors.mean()),
                       'typical_error_recent': np.exp(errors[last].mean()),
                       'peak_week_right': g['peak_week_right'].mean(),
                       'peak_week_within_1': g['peak_week_within_1'].mean()})
scores = pd.DataFrame(score_rows)
headline = scores.loc[scores.groupby('area')['typical_error_recent'].idxmin(), ['area', 'method']]
HEADLINE = dict(zip(headline['area'], headline['method']))
display(scores.round(2))
for area, method in HEADLINE.items():
    print(f'{area}: headline method "{method}".')
    if frozen:
        saved = saved_summary.loc[(saved_summary['area'] == area) & saved_summary['headline'], 'method']
        if len(saved) and saved.iloc[0] != method:
            print(f'  The saved forecast kept "{saved.iloc[0]}", the headline when it was issued.')

shown = BACKTEST_SEASONS[-2:]
fig, axes = plt.subplots(len(AREAS), len(shown), figsize=(11, 3.6 * len(AREAS)), squeeze=False)
for i, area in enumerate(AREAS):
    for j, season in enumerate(shown):
        ax = axes[i, j]
        ax.plot(range(1, WEEKS + 1), snpp[area].loc[season], color='k', marker='o', label='what happened')
        for name in METHODS:
            ax.plot(range(ISSUE_WEEK + 1, WEEKS + 1), backtest_weekly[area, season, name], color=COLOURS[name],
                    marker='.', label=name)
        ax.axvline(ISSUE_WEEK + 0.5, color='k', linestyle=':', linewidth=1)
        ax.set_title(f'{area}, {season}: forecast after {week_label(season, ISSUE_WEEK)}', fontsize=9)
        ax.set_xticks(range(1, WEEKS + 1))
        ax.set_ylabel('Suomi NPP detections per week')
axes[0, 0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / '02_backtests.png', dpi=150)
plt.show()

## 5. The 2026 forecast

Made once, from the weeks of 2026 seen so far, and saved with its date. Later runs load it instead of forecasting again, so the check in section 6 scores what was actually forecast. To replace it (for example after fixing an error in this notebook), set `REISSUE = True`, run, then set it back.

In [ ]:
if frozen:
    forecast = pd.read_csv(FORECAST_FILE)
    summary = saved_summary
    print(f"Forecast issued on {summary['issued_on'].iloc[0]} with data to {summary['data_through'].iloc[0]} "
          '(loaded, not re-made).')
else:
    issued_on, data_through = TODAY.strftime('%Y-%m-%d'), week_dates(SEASON, ISSUE_WEEK)[1].strftime('%Y-%m-%d')
    weekly_rows, summary_rows = [], []
    for area in AREAS:
        past = snpp[area].loc[FIRST_SEASON:SEASON - 1].to_numpy(float)
        seen = snpp[area].loc[SEASON].to_numpy(float)[:ISSUE_WEEK]
        for name, method in METHODS.items():
            s = summarise(method(past, seen, ISSUE_WEEK), seen)
            # The range: this forecast scaled by the spread of the method's past errors (all backtest seasons).
            past_errors = -log_ratio(*backtests.loc[(backtests['area'] == area) & (backtests['method'] == name),
                                                    ['forecast_rest', 'actual_rest']].to_numpy().T)
            low, high = np.quantile(past_errors, [(1 - INTERVAL) / 2, (1 + INTERVAL) / 2])
            score = scores[(scores['area'] == area) & (scores['method'] == name)].iloc[0]
            summary_rows.append({
                'area': area, 'method': name, 'headline': HEADLINE[area] == name, 'issued_on': issued_on,
                'data_through': data_through, 'issue_week': ISSUE_WEEK, 'seen_so_far': seen.sum(),
                'forecast_rest': round(s['rest']), 'rest_low': round(max((s['rest'] + 1) * np.exp(low) - 1, 0)),
                'rest_high': round((s['rest'] + 1) * np.exp(high) - 1), 'interval': INTERVAL,
                'forecast_season': round(seen.sum() + s['rest']), 'last_season': snpp[area].loc[SEASON - 1].sum(),
                'peak_week': s['peak_week'], 'peak_week_dates': week_label(SEASON, s['peak_week']),
                'peak_week_share': round(s['peak_share'], 2), 'peak_height': round(s['peak_height']),
                'typical_error_all': round(score['typical_error_all'], 2),
                'typical_error_recent': round(score['typical_error_recent'], 2)})
            for week in range(1, WEEKS + 1):
                start, end = week_dates(SEASON, week)
                weekly_rows.append({'area': area, 'method': name, 'headline': HEADLINE[area] == name,
                                    'issued_on': issued_on, 'week': week, 'week_start': start.strftime('%Y-%m-%d'),
                                    'week_end': end.strftime('%Y-%m-%d'),
                                    'seen': seen[week - 1] if week <= ISSUE_WEEK else np.nan,
                                    'forecast': round(s['weekly'][week - ISSUE_WEEK - 1], 1) if week > ISSUE_WEEK
                                    else np.nan})
    forecast, summary = pd.DataFrame(weekly_rows), pd.DataFrame(summary_rows)
    forecast.to_csv(FORECAST_FILE, index=False)
    summary.to_csv(SUMMARY_FILE, index=False)
    print(f'Forecast issued on {issued_on} from weeks 1-{ISSUE_WEEK} (data to {data_through}); saved.')

display(summary[['area', 'method', 'headline', 'forecast_rest', 'rest_low', 'rest_high', 'forecast_season',
                 'last_season', 'peak_week_dates', 'peak_week_share', 'peak_height', 'typical_error_recent']])
for _, row in summary[summary['headline']].iterrows():
    seen_text = f"{row['seen_so_far']:,.0f} so far"
    share = ('its only scenario' if row['method'] == 'last season'
             else f"{row['peak_week_share']:.0%} of its scenarios")
    print(f"{row['area']} ({row['method']}): busiest week most likely {row['peak_week_dates']} "
          f"({share}), about {row['peak_height']:,.0f} detections in that week. "
          f"About {row['forecast_rest']:,.0f} more detections to 30 Nov ({row['interval']:.0%} range "
          f"{row['rest_low']:,.0f}-{row['rest_high']:,.0f}); {seen_text}, about {row['forecast_season']:,.0f} for the "
          f"season, against {row['last_season']:,.0f} in {SEASON - 1}.")

fig, axes = plt.subplots(1, len(AREAS), figsize=(12, 4.2))
for ax, area in zip(np.atleast_1d(axes), AREAS):
    issue = int(summary['issue_week'].iloc[0])
    past = snpp[area].loc[FIRST_SEASON:SEASON - 1].to_numpy(float)
    seen = forecast[(forecast['area'] == area) & (forecast['method'] == 'last season')]['seen'].to_numpy()[:issue]
    for scenario in past_seasons_rescaled(past, seen, issue):
        ax.plot(range(issue, WEEKS + 1), np.r_[seen[-1], scenario], color='0.85', linewidth=0.8)
    ax.plot(range(1, issue + 1), seen, color='k', marker='o', linewidth=2, label=f'{SEASON}, seen')
    for name in METHODS:
        f = forecast[(forecast['area'] == area) & (forecast['method'] == name)]['forecast'].to_numpy()[issue:]
        is_headline = bool(summary.loc[(summary['area'] == area) & (summary['method'] == name), 'headline'].iloc[0])
        ax.plot(range(issue, WEEKS + 1), np.r_[seen[-1], f], color=COLOURS[name], marker='.',
                linewidth=2.4 if is_headline else 1.2, label=name + (' (headline)' if is_headline else ''))
    ax.axvline(issue + 0.5, color='k', linestyle=':', linewidth=1)
    ax.set_xticks(range(1, WEEKS + 1))
    ax.set_xticklabels([day_month(week_dates(SEASON, w)[0]) for w in range(1, WEEKS + 1)], rotation=45, fontsize=8)
    ax.set_title(f'{area}: forecast issued {summary["issued_on"].iloc[0]} (grey: each past season rescaled)',
                 fontsize=10)
    ax.set_ylabel('Suomi NPP detections per week')
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGS / '03_forecast_2026.png', dpi=150)
plt.show()

## 6. Checking the forecast as the season ends

Each run compares the forecast with the weeks that have ended. Up to week 7 (to 2 Nov) the actual count is Suomi NPP's own. From week 8, Suomi NPP has stopped, so the actual count is NOAA-20's, multiplied by the ratio of the two sensors over the 2026 weeks both covered (section 3). The ratio varied between seasons, which adds a few percent of uncertainty to those weeks; re-run with a ratio from the 2020–2025 range to see how much.

The rest-of-season score is final once week 11 has ended (after 30 Nov).

In [ ]:
check_rows = []
for area in AREAS:
    for week in range(int(summary['issue_week'].iloc[0]) + 1, WEEKS + 1):
        start, end = week_dates(SEASON, week)
        if complete['SNPP'][week]:
            actual, source = snpp[area].loc[SEASON, week], 'Suomi NPP'
        elif complete['NOAA20'][week]:
            actual, source = noaa20[area].loc[SEASON, week] * ratio_2026, f'NOAA-20 x {ratio_2026:.2f}'
        else:
            actual, source = np.nan, 'not yet'
        row = {'area': area, 'week': week, 'week_start': start.strftime('%Y-%m-%d'),
               'week_end': end.strftime('%Y-%m-%d'), 'actual': round(actual, 1) if np.isfinite(actual) else np.nan,
               'actual_source': source}
        for name in METHODS:
            row[name] = forecast.loc[(forecast['area'] == area) & (forecast['method'] == name)
                                     & (forecast['week'] == week), 'forecast'].iloc[0]
        check_rows.append(row)
check = pd.DataFrame(check_rows)
check.to_csv(PROCESSED / 'fire_forecast_check_2026.csv', index=False)
ended = check['actual'].notna()
if not ended.any():
    first_end = week_dates(SEASON, int(summary['issue_week'].iloc[0]) + 1)[1]
    print(f'No forecast week has ended yet: re-run after {day_month(first_end)} to start the check.')
else:
    display(check[ended])
    for area in AREAS:
        part = check[(check['area'] == area)]
        if part['actual'].notna().all():
            row = summary[(summary['area'] == area) & summary['headline']].iloc[0]
            actual_rest = part['actual'].sum()
            inside = row['rest_low'] <= actual_rest <= row['rest_high']
            seen = forecast[(forecast['area'] == area) & (forecast['method'] == row['method'])]['seen'].dropna()
            peak = int(np.argmax(np.r_[seen.to_numpy(), part['actual'].to_numpy()])) + 1
            print(f"{area}: {actual_rest:,.0f} detections after week {row['issue_week']} against "
                  f"{row['forecast_rest']:,.0f} forecast by {row['method']} "
                  f"({'inside' if inside else 'outside'} its range {row['rest_low']:,.0f}-{row['rest_high']:,.0f}). "
                  f"Busiest week: {week_label(SEASON, peak)}, forecast {row['peak_week_dates']}.")
        else:
            print(f'{area}: {part["actual"].notna().sum()} of {len(part)} forecast weeks have ended.')

    fig, axes = plt.subplots(1, len(AREAS), figsize=(12, 4))
    for ax, area in zip(np.atleast_1d(axes), AREAS):
        part = check[check['area'] == area]
        for name in METHODS:
            ax.plot(part['week'], part[name], color=COLOURS[name], marker='.', label=name)
        for source, marker in (('Suomi NPP', 'o'), ('NOAA-20', 's')):
            sel = part['actual_source'].str.startswith(source)
            ax.scatter(part.loc[sel, 'week'], part.loc[sel, 'actual'], color='k', marker=marker, zorder=3,
                       label=f'actual ({source}{" converted" if source == "NOAA-20" else ""})')
        ax.set_title(f'{area}: forecast against what happened', fontsize=10)
        ax.set_xticks(part['week'].tolist())               # whole weeks only
        ax.set_xlabel('week of the season')
        ax.set_ylabel('detections per week (Suomi NPP terms)')
    np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGS / '04_forecast_check.png', dpi=150)
    plt.show()

## 7. Outputs

In [ ]:
rows = []
for area in AREAS:
    for sensor, tables in (('SNPP', snpp), ('NOAA20', noaa20)):
        long = tables[area].stack().rename('detections').reset_index()
        long.columns = ['season', 'week', 'detections']
        long['complete'] = True
        is_now = long['season'] == SEASON
        long.loc[is_now, 'complete'] = long.loc[is_now, 'week'].map(complete[sensor]).to_numpy()
        data_end = snpp_end if sensor == 'SNPP' else last_full_day
        no_data = is_now & long['week'].map(lambda w: week_dates(SEASON, w)[0] > data_end)
        long.loc[no_data, 'detections'] = np.nan          # e.g. Suomi NPP after 2 Nov: no data, not zero fires
        rows.append(long.assign(area=area, sensor=SENSOR_NAME[sensor]))
counts = pd.concat(rows, ignore_index=True)
counts['week_start'] = [week_dates(s, w)[0].strftime('%Y-%m-%d') for s, w in zip(counts['season'], counts['week'])]
counts['product'] = np.where(counts['season'] == SEASON, 'NRT', 'SP')
counts = counts[['area', 'sensor', 'product', 'season', 'week', 'week_start', 'detections', 'complete']]
counts.to_csv(PROCESSED / 'fire_weekly_counts.csv', index=False)
backtests.to_csv(PROCESSED / 'fire_forecast_backtests.csv', index=False)

for source, sensor in SOURCE_SENSOR.items():
    end = snpp_end if sensor == 'SNPP' else last_full_day
    register_source(
        source_id=f'firms_{source.lower()}_2026_forecast',
        dataset_or_product=f'VIIRS 375 m active fire detections, {SENSOR_NAME[sensor]}, near real time (NRT); '
                           f'FIRMS API source {source}',
        provider='NASA FIRMS (Fire Information for Resource Management System)',
        url='https://firms.modaps.eosdis.nasa.gov/api/area/',
        date_accessed=TODAY.strftime('%Y-%m-%d'),
        geography=f'Bounding box {BBOX}, clipped to Punjab and Haryana districts',
        time_range=f'{season_start(SEASON):%Y-%m-%d} to {end:%Y-%m-%d}',
        resolution='375 m fire pixels; detections only at satellite overpass times',
        licence_or_terms='NASA open data policy; cite per the FIRMS citation policy',
        preprocessing="Notebook 01's cleaning rules; counted by IST week from 15 Sep",
        known_limitations='Near-real-time data can differ slightly from the later standard processing; Suomi NPP '
                          'data stop at 13:00 UTC on 2 Nov 2026',
        used_in='notebooks/04_fire_forecast.ipynb')
print('Outputs written to', PROCESSED, 'and', FIGS)

### Reading the results
- **The forecast is fixed once made.** Commit `fire_forecast_2026.csv` and its summary soon after the first run: the commit date shows the forecast came before the outcome.
- **Typical error factor.** A factor of 1.5 means the method's rest-of-season totals were typically 1.5 times too high or too low in the backtests. In the seasons when detections collapsed, methods that lean on past levels overshot badly; that is why the headline is chosen on the last five seasons.
- **The peak week** is the week most scenarios peak in. A share of 60% means 60% of the scenarios (SARIMA's paths, or the past seasons rescaled) put the peak there; it is a guide, not a calibrated probability.

### Caveats
- **Detections, not burnt area.** Satellites pass in the early afternoon; fires lit later are missed (Notebook 01), and small or brief fires too.
- **A changing sensor.** After 2 Nov the check depends on NOAA-20 and a conversion ratio; the 2020–2025 seasons put that ratio between about 0.87 and 1.01.
- **Near real time.** 2026 counts are near-real-time data; the later standard processing can differ slightly from them, and from the archive the methods learnt from.
- **Ten backtests.** The 80% range rests on ten past seasons, so it is rough; treat it as a guide to the size of the uncertainty.
- **One state and one district.** Sangrur's weekly counts are small, so its forecast is noisier than Punjab's.